In [1]:
import psycopg2

USER = "postgres"
PASSWORD = "Mkilo1990"

#Connexion à la base store_db
conn = psycopg2.connect(
    dbname="megabase0",
    user=USER,
    password=PASSWORD,
    host="localhost",
    port="5432"
)
conn.autocommit = True


# 1.1 Nombre de lignes dans chaque table

cur = conn.cursor()

cur.execute("""
    SELECT 'commune' AS table_name, COUNT(*) FROM commune
    UNION ALL SELECT 'departement', COUNT(*) FROM departement
    UNION ALL SELECT 'region', COUNT(*) FROM region
    UNION ALL SELECT 'lycee', COUNT(*) FROM lycee
    UNION ALL SELECT 'college', COUNT(*) FROM college
    UNION ALL SELECT 'pharmacie', COUNT(*) FROM pharmacie
    UNION ALL SELECT 'ehpad', COUNT(*) FROM ehpad
    UNION ALL SELECT 'bibliotheque', COUNT(*) FROM bibliotheque
    UNION ALL SELECT 'mairie', COUNT(*) FROM mairie
    UNION ALL SELECT 'entreprise_btp', COUNT(*) FROM entreprise_btp
    UNION ALL SELECT 'festivals', COUNT(*) FROM festivals
    ORDER BY table_name;
""")

rows = cur.fetchall()
for r in rows:
    print(r)

('bibliotheque', 15674)
('college', 9157)
('commune', 34968)
('departement', 108)
('ehpad', 7422)
('entreprise_btp', 9284)
('festivals', 6677)
('lycee', 5578)
('mairie', 34968)
('pharmacie', 20454)
('region', 25)


In [2]:
# 1.2 Nombre de communes par région

cur.execute("""
SELECT r.name AS region,
       COUNT(*) AS nb_communes
FROM commune c
JOIN departement d ON c.code_departement = d.code_departement
JOIN region r ON d.code_region = r.code_region
GROUP BY r.name
ORDER BY nb_communes DESC;
""")
rows = cur.fetchall()
for r in rows:
    print(r)

('Grand Est', 5115)
('Occitanie', 4446)
('Nouvelle-Aquitaine', 4293)
('Auvergne-Rhône-Alpes', 4025)
('Hauts-de-France', 3782)
('Bourgogne-Franche-Comté', 3685)
('Normandie', 2644)
('Centre-Val de Loire', 1754)
('Île-de-France', 1266)
('Pays de la Loire', 1228)
('Bretagne', 1202)
("Provence-Alpes-Côte d'Azur", 946)
('Corse', 360)
('Polynésie française', 48)
('Martinique', 34)
('Nouvelle-Calédonie', 33)
('Guadeloupe', 32)
('La Réunion', 24)
('Guyane', 22)
('Mayotte', 17)
('Terres australes et antarctiques françaises', 5)
('Wallis et Futuna', 3)
('Saint-Pierre-et-Miquelon', 2)
('Saint-Martin', 1)
('Saint-Barthélemy', 1)


In [3]:
# 1.3 Nombre de communes par département

cur.execute("""
SELECT d.name AS departement,
       COUNT(*) AS nb_communes
FROM commune c
JOIN departement d ON c.code_departement = d.code_departement
GROUP BY d.name
ORDER BY nb_communes DESC;
""")
rows = cur.fetchall()
for r in rows:
    print(r)

('Pas-de-Calais', 887)
('Aisne', 797)
('Somme', 771)
('Moselle', 725)
('Seine-Maritime', 707)
("Côte-d'Or", 698)
('Oise', 680)
('Nord', 647)
('Marne', 610)
('Meurthe-et-Moselle', 591)
('Haute-Garonne', 586)
('Eure', 585)
('Doubs', 563)
('Saône-et-Loire', 563)
('Pyrénées-Atlantiques', 545)
('Haute-Saône', 536)
('Gironde', 534)
('Calvados', 526)
('Bas-Rhin', 514)
('Isère', 512)
('Seine-et-Marne', 507)
('Vosges', 506)
('Dordogne', 503)
('Meuse', 499)
('Jura', 492)
('Hautes-Pyrénées', 469)
('Puy-de-Dôme', 463)
('Charente-Maritime', 462)
('Gers', 458)
('Ardennes', 447)
('Manche', 445)
('Aude', 433)
('Aube', 431)
('Haute-Marne', 426)
('Yonne', 423)
('Ain', 391)
('Orne', 381)
('Haut-Rhin', 366)
('Eure-et-Loir', 363)
('Drôme', 362)
('Charente', 359)
('Sarthe', 352)
('Gard', 350)
("Côtes-d'Armor", 344)
('Hérault', 341)
('Ardèche', 335)
('Ille-et-Vilaine', 332)
('Landes', 327)
('Loiret', 325)
('Ariège', 325)
('Loire', 320)
('Lot-et-Garonne', 319)
('Allier', 317)
('Tarn', 314)
('Lot', 312)
('Nièv

In [4]:
# 1.4 le nombre d'établissements d'un type (pharmacies, lycées...) **par département**

cur.execute("""
            
SELECT d.name AS departement,

    COALESCE(bib.nb, 0) AS nb_bibliotheques,
    COALESCE(col.nb, 0) AS nb_colleges,
    COALESCE(ehp.nb, 0) AS nb_ehpads,
    COALESCE(btp.nb, 0) AS nb_entreprises_btp,
    COALESCE(fes.nb, 0) AS nb_festivals,
    COALESCE(lyc.nb, 0) AS nb_lycees,
    COALESCE(mai.nb, 0) AS nb_mairies,
    COALESCE(pha.nb, 0) AS nb_pharmacies

FROM departement d

LEFT JOIN (SELECT c.code_departement, COUNT(*) AS nb  FROM bibliotheque b  JOIN commune c ON b.insee_code = c.insee_code  GROUP BY c.code_departement) bib ON bib.code_departement = d.code_departement

LEFT JOIN (SELECT c.code_departement, COUNT(*) AS nb  FROM college col  JOIN commune c ON col.insee_code = c.insee_code  GROUP BY c.code_departement) col ON col.code_departement = d.code_departement

LEFT JOIN (SELECT c.code_departement, COUNT(*) AS nb  FROM ehpad e  JOIN commune c ON e.insee_code = c.insee_code  GROUP BY c.code_departement) ehp ON ehp.code_departement = d.code_departement

LEFT JOIN (SELECT c.code_departement, COUNT(*) AS nb  FROM entreprise_btp b  JOIN commune c ON b.insee_code = c.insee_code  GROUP BY c.code_departement) btp ON btp.code_departement = d.code_departement

LEFT JOIN (SELECT c.code_departement, COUNT(*) AS nb  FROM festivals f  JOIN commune c ON f.insee_code = c.insee_code  GROUP BY c.code_departement) fes ON fes.code_departement = d.code_departement

LEFT JOIN (SELECT c.code_departement, COUNT(*) AS nb  FROM lycee l  JOIN commune c ON l.insee_code = c.insee_code GROUP BY c.code_departement) lyc ON lyc.code_departement = d.code_departement

LEFT JOIN (SELECT c.code_departement, COUNT(*) AS nb  FROM mairie m  JOIN commune c ON m.insee_code = c.insee_code  GROUP BY c.code_departement) mai ON mai.code_departement = d.code_departement

LEFT JOIN (SELECT c.code_departement, COUNT(*) AS nb  FROM pharmacie p  JOIN commune c ON p.insee_code = c.insee_code  GROUP BY c.code_departement) pha ON pha.code_departement = d.code_departement

ORDER BY departement;
""")

rows = cur.fetchall()
for r in rows:
    print(r)

('Ain', 230, 78, 66, 0, 0, 43, 391, 157)
('Aisne', 129, 92, 67, 0, 0, 51, 797, 165)
('Allier', 215, 51, 48, 0, 0, 32, 317, 126)
('Alpes-de-Haute-Provence', 94, 23, 32, 0, 0, 15, 198, 59)
('Alpes-Maritimes', 126, 122, 150, 0, 0, 79, 163, 428)
('Ardèche', 218, 47, 65, 0, 0, 24, 335, 97)
('Ardennes', 100, 49, 31, 0, 0, 24, 447, 102)
('Ariège', 76, 23, 32, 0, 0, 18, 325, 48)
('Aube', 144, 43, 44, 0, 21, 26, 431, 87)
('Aude', 251, 44, 56, 0, 95, 34, 433, 135)
('Aveyron', 187, 47, 68, 0, 37, 33, 285, 106)
('Bas-Rhin', 215, 144, 116, 0, 101, 86, 514, 273)
('Bouches-du-Rhône', 141, 242, 196, 0, 304, 173, 119, 733)
('Calvados', 131, 88, 89, 0, 82, 61, 526, 206)
('Cantal', 153, 29, 39, 0, 21, 17, 250, 65)
('Charente', 75, 59, 72, 0, 52, 30, 359, 121)
('Charente-Maritime', 224, 76, 117, 0, 98, 44, 462, 216)
('Cher', 148, 44, 42, 0, 38, 27, 286, 99)
('Corrèze', 117, 36, 44, 0, 37, 30, 277, 92)
('Corse-du-Sud', 49, 20, 14, 0, 33, 12, 124, 60)
("Côte-d'Or", 208, 73, 76, 0, 66, 38, 698, 162)
("Côtes-

In [5]:
# 2.1 Population totale par région

cur.execute("""
            
SELECT r.name AS region,
       SUM(c.population) AS population_totale
FROM commune c
JOIN departement d ON c.code_departement = d.code_departement
JOIN region r ON d.code_region = r.code_region
GROUP BY r.name
ORDER BY population_totale DESC;
""")
rows = cur.fetchall()
for r in rows:
    print(r)

('Terres australes et antarctiques françaises', None)
('Île-de-France', 12463067)
('Auvergne-Rhône-Alpes', 8205557)
('Nouvelle-Aquitaine', 6150451)
('Occitanie', 6124653)
('Hauts-de-France', 5992194)
('Grand Est', 5563378)
("Provence-Alpes-Côte d'Azur", 5218960)
('Pays de la Loire', 3907156)
('Bretagne', 3449370)
('Normandie', 3345842)
('Bourgogne-Franche-Comté', 2802670)
('Centre-Val de Loire', 2587031)
('La Réunion', 889679)
('Guadeloupe', 384160)
('Martinique', 360630)
('Corse', 355486)
('Guyane', 293996)
('Polynésie française', 278786)
('Nouvelle-Calédonie', 264596)
('Mayotte', 256518)
('Saint-Martin', 31160)
('Wallis et Futuna', 11151)
('Saint-Barthélemy', 10660)
('Saint-Pierre-et-Miquelon', 5790)


In [6]:
# 2.2 Classement des départements par nombre de lycées

cur.execute("""
SELECT d.name AS departement,
       COUNT(*) AS nb_lycees
FROM lycee l
JOIN commune c ON l.insee_code = c.insee_code
JOIN departement d ON c.code_departement = d.code_departement
GROUP BY d.name
ORDER BY nb_lycees DESC;
        
""")
rows = cur.fetchall()
for r in rows:
    print(r)

('Paris', 232)
('Nord', 184)
('Bouches-du-Rhône', 173)
('Rhône', 155)
('Seine-Saint-Denis', 146)
('Loire-Atlantique', 122)
('Haute-Garonne', 117)
('Yvelines', 116)
('Gironde', 114)
('Seine-et-Marne', 105)
('Seine-Maritime', 105)
('Val-de-Marne', 103)
('Hauts-de-Seine', 101)
('Hérault', 101)
("Val-d'Oise", 91)
('Pas-de-Calais', 88)
('Isère', 88)
('Bas-Rhin', 86)
('Ille-et-Vilaine', 85)
('Moselle', 82)
('La Réunion', 82)
('Finistère', 81)
('Alpes-Maritimes', 79)
('Loire', 78)
('Essonne', 78)
('Pyrénées-Atlantiques', 75)
('Maine-et-Loire', 68)
('Var', 62)
('Calvados', 61)
('Morbihan', 60)
('Haut-Rhin', 59)
('Vendée', 58)
('Meurthe-et-Moselle', 57)
('Oise', 56)
('Puy-de-Dôme', 55)
('Somme', 54)
('Gard', 54)
('Haute-Savoie', 53)
('Loiret', 53)
('Aisne', 51)
('Sarthe', 50)
("Côtes-d'Armor", 50)
('Guadeloupe', 47)
('Martinique', 47)
('Marne', 46)
('Vaucluse', 46)
('Indre-et-Loire', 46)
('Doubs', 46)
('Eure', 46)
('Charente-Maritime', 44)
('Ain', 43)
('Manche', 41)
('Tarn', 40)
('Vosges', 39)


In [8]:
#2.3 Moyenne d’établissements (pharmacies) par commune dans un département

cur.execute("""
            
SELECT d.name AS departement,
       AVG(ph.nb) AS moyenne_pharmacies_par_commune
FROM (
    SELECT c.insee_code, COUNT(*) AS nb
    FROM pharmacie p
    JOIN commune c ON p.insee_code = c.insee_code
    GROUP BY c.insee_code
) ph
JOIN commune c ON ph.insee_code = c.insee_code
JOIN departement d ON c.code_departement = d.code_departement
GROUP BY d.name
ORDER BY moyenne_pharmacies_par_commune DESC;
""")

rows = cur.fetchall()
for r in rows:
    print(r)

('Paris', Decimal('874.0000000000000000'))
('Hauts-de-Seine', Decimal('12.9142857142857143'))
('Saint-Martin', Decimal('11.0000000000000000'))
('La Réunion', Decimal('10.2083333333333333'))
('Seine-Saint-Denis', Decimal('9.8974358974358974'))
('Val-de-Marne', Decimal('8.2173913043478261'))
('Bouches-du-Rhône', Decimal('7.2574257425742574'))
('Alpes-Maritimes', Decimal('6.3880597014925373'))
('Guadeloupe', Decimal('4.8333333333333333'))
('Martinique', Decimal('4.6896551724137931'))
('Guyane', Decimal('4.2500000000000000'))
('Rhône', Decimal('4.0687022900763359'))
("Val-d'Oise", Decimal('3.7976190476190476'))
('Var', Decimal('3.7395833333333333'))
('Yvelines', Decimal('3.3333333333333333'))
('Essonne', Decimal('3.2038834951456311'))
('Haute-Garonne', Decimal('3.1259842519685039'))
('Saint-Barthélemy', Decimal('3.0000000000000000'))
('Hérault', Decimal('2.8656716417910448'))
('Nord', Decimal('2.7777777777777778'))
('Gironde', Decimal('2.7379679144385027'))
('Vaucluse', Decimal('2.67123287

In [10]:
# 3. CROISER LES SOURCES


#3.1 Communes qui ont un lycée mais aucune pharmacie

cur.execute("""
            
SELECT c.name AS commune,
       d.name AS departement
FROM commune c
JOIN departement d ON c.code_departement = d.code_departement
LEFT JOIN lycee l ON c.insee_code = l.insee_code
LEFT JOIN pharmacie p ON c.insee_code = p.insee_code
WHERE l.uai IS NOT NULL
  AND p.finess IS NULL
ORDER BY d.name, c.name;

""")

rows = cur.fetchall()
for r in rows:
    print(r)

('Saint-Sorlin-en-Bugey', 'Ain')
('Coucy-la-Ville', 'Aisne')
('Épaux-Bézu', 'Aisne')
('Fontaine-lès-Vervins', 'Aisne')
('Fontaine-lès-Vervins', 'Aisne')
('Fontaine-lès-Vervins', 'Aisne')
('Le Hérie-la-Viéville', 'Aisne')
('Durdat-Larequille', 'Allier')
('Toulon-sur-Allier', 'Allier')
('Le Chaffaut-Saint-Jurson', 'Alpes-de-Haute-Provence')
('Valdeblore', 'Alpes-Maritimes')
('Valdeblore', 'Alpes-Maritimes')
('Saint-Laurent', 'Ardennes')
('Ferrières-sur-Ariège', 'Ariège')
('Ferrières-sur-Ariège', 'Ariège')
('Moulis', 'Ariège')
('Les Loges-Margueron', 'Aube')
('Saint-Pouange', 'Aube')
('Sainte-Maure', 'Aube')
('Souilhanels', 'Aude')
('Monteils', 'Aveyron')
('Urmatt', 'Bas-Rhin')
('Walbourg', 'Bas-Rhin')
('Firfol', 'Calvados')
('Saint-Manvieu-Norrey', 'Calvados')
('Salles-de-Barbezieux', 'Charente')
('Verrières', 'Charente')
('Bois', 'Charente-Maritime')
('Pessines', 'Charente-Maritime')
('Saint-Germain-de-Lusignan', 'Charente-Maritime')
('Bengy-sur-Craon', 'Cher')
('Le Subdray', 'Cher')
('

In [11]:
#3.2 Profil de service d’une commune (lycées, collèges, pharmacies, ehpad)
cur.execute("""
                        
SELECT c.name AS commune,
       d.name AS departement,
       COALESCE(l.nb_lycees, 0) AS lycees,
       COALESCE(co.nb_colleges, 0) AS colleges,
       COALESCE(ph.nb_pharmacies, 0) AS pharmacies,
       COALESCE(eh.nb_ehpad, 0) AS ehpad
FROM commune c
JOIN departement d ON c.code_departement = d.code_departement

LEFT JOIN (
    SELECT insee_code, COUNT(*) AS nb_lycees
    FROM lycee GROUP BY insee_code
) l ON c.insee_code = l.insee_code

LEFT JOIN (
    SELECT insee_code, COUNT(*) AS nb_colleges
    FROM college GROUP BY insee_code
) co ON c.insee_code = co.insee_code

LEFT JOIN (
    SELECT insee_code, COUNT(*) AS nb_pharmacies
    FROM pharmacie GROUP BY insee_code
) ph ON c.insee_code = ph.insee_code

LEFT JOIN (
    SELECT insee_code, COUNT(*) AS nb_ehpad
    FROM ehpad GROUP BY insee_code
) eh ON c.insee_code = eh.insee_code

ORDER BY d.name, c.name; 
            
            """)
rows = cur.fetchall()
for r in rows:
    print(r)

('Ambérieu-en-Bugey', 'Ain', 2, 2, 4, 2)
('Ambérieux-en-Dombes', 'Ain', 0, 0, 1, 0)
('Ambléon', 'Ain', 0, 0, 0, 0)
('Ambronay', 'Ain', 0, 0, 1, 0)
('Ambutrix', 'Ain', 0, 0, 0, 0)
('Andert-et-Condon', 'Ain', 0, 0, 0, 0)
('Anglefort', 'Ain', 0, 0, 0, 0)
('Apremont', 'Ain', 0, 0, 0, 0)
('Aranc', 'Ain', 0, 0, 0, 0)
('Arandas', 'Ain', 0, 0, 0, 0)
('Arbent', 'Ain', 1, 1, 1, 0)
('Arbigny', 'Ain', 0, 0, 0, 0)
('Arboys en Bugey', 'Ain', 0, 0, 0, 0)
('Argis', 'Ain', 0, 0, 0, 0)
('Armix', 'Ain', 0, 0, 0, 0)
('Ars-sur-Formans', 'Ain', 0, 0, 1, 0)
('Artemare', 'Ain', 0, 1, 1, 0)
('Arvière-en-Valromey', 'Ain', 0, 0, 0, 0)
('Asnières-sur-Saône', 'Ain', 0, 0, 0, 0)
('Attignat', 'Ain', 0, 0, 1, 0)
('Bâgé-Dommartin', 'Ain', 0, 1, 1, 0)
('Bâgé-le-Châtel', 'Ain', 0, 0, 0, 0)
('Balan', 'Ain', 0, 0, 0, 0)
('Baneins', 'Ain', 0, 0, 0, 0)
('Béard-Géovreissiat', 'Ain', 0, 0, 0, 0)
('Beaupont', 'Ain', 0, 0, 0, 0)
('Beauregard', 'Ain', 0, 0, 0, 0)
('Béligneux', 'Ain', 0, 0, 1, 1)
('Belley', 'Ain', 4, 3, 4, 3)
('B

In [12]:
#4. INDICATEURS


 #4.1 Habitants par pharmacie (zones sous-dotées)

cur.execute("""
            
SELECT d.name AS departement,
       SUM(c.population) AS population_totale,
       COUNT(p.finess) AS nb_pharmacies,
       ROUND(SUM(c.population)::numeric / NULLIF(COUNT(p.finess),0), 2)
           AS habitants_par_pharmacie
FROM commune c
JOIN departement d ON c.code_departement = d.code_departement
LEFT JOIN pharmacie p ON c.insee_code = p.insee_code
GROUP BY d.name
ORDER BY habitants_par_pharmacie DESC;
            """)

rows = cur.fetchall()
for r in rows:
    print(r)


('Nouvelle-Calédonie', 264596, 0, None)
('Wallis et Futuna', 11151, 0, None)
('Terres australes et antarctiques françaises', None, 0, None)
('Polynésie française', 278786, 0, None)
('Paris', 1838701972, 874, Decimal('2103778.00'))
('Bouches-du-Rhône', 331400592, 733, Decimal('452115.41'))
('Haute-Garonne', 82711591, 397, Decimal('208341.54'))
('Rhône', 95988676, 533, Decimal('180091.32'))
('Alpes-Maritimes', 64416222, 428, Decimal('150505.19'))
('Hérault', 35472105, 384, Decimal('92375.27'))
('Bas-Rhin', 25037644, 273, Decimal('91712.98'))
('Loire-Atlantique', 32588736, 389, Decimal('83775.67'))
('La Réunion', 18195764, 245, Decimal('74268.42'))
('Gironde', 36566084, 512, Decimal('71418.13'))
('Marne', 12619064, 178, Decimal('70893.62'))
('Seine-Saint-Denis', 24035592, 386, Decimal('62268.37'))
('Hauts-de-Seine', 27782189, 452, Decimal('61465.02'))
('Var', 20533166, 359, Decimal('57195.45'))
("Côte-d'Or", 9115411, 162, Decimal('56267.97'))
('Ille-et-Vilaine', 15939352, 288, Decimal('55

In [13]:
#  4.2 Taux d’inscription aux bibliothèques
cur.execute("""
SELECT d.name AS departement,
       SUM(b.population) AS population_totale,
       SUM(b.borrowers) AS emprunteurs,
       ROUND(SUM(b.borrowers)::numeric / NULLIF(SUM(b.population),0), 4)
           AS taux_inscription
FROM bibliotheque b
JOIN commune c ON b.insee_code = c.insee_code
JOIN departement d ON c.code_departement = d.code_departement
GROUP BY d.name
ORDER BY taux_inscription DESC;

""")

rows = cur.fetchall()
for r in rows:
    print(r)

('Nouvelle-Calédonie', None, 1359, None)
('Saint-Barthélemy', 10556, None, None)
('Saint-Martin', 32010, None, None)
('Mayotte', 225771, 70449, Decimal('0.3120'))
('Ariège', 106237, 23829, Decimal('0.2243'))
('Hautes-Alpes', 132757, 28369, Decimal('0.2137'))
('Ardèche', 288698, 45696, Decimal('0.1583'))
('Lozère', 71619, 10876, Decimal('0.1519'))
('Landes', 340912, 50301, Decimal('0.1475'))
('Vosges', 251505, 35254, Decimal('0.1402'))
('Lot', 148507, 20669, Decimal('0.1392'))
('Cantal', 136902, 18882, Decimal('0.1379'))
('Haute-Loire', 249182, 33916, Decimal('0.1361'))
('Creuse', 82322, 11003, Decimal('0.1337'))
('Haute-Saône', 140144, 18200, Decimal('0.1299'))
('Aveyron', 286507, 36557, Decimal('0.1276'))
('Corrèze', 244845, 30590, Decimal('0.1249'))
('Savoie', 521073, 63500, Decimal('0.1219'))
('Drôme', 736755, 87008, Decimal('0.1181'))
('Nièvre', 147247, 17354, Decimal('0.1179'))
('Mayenne', 387344, 43599, Decimal('0.1126'))
('Ain', 728570, 79488, Decimal('0.1091'))
('Gers', 152102,

In [5]:
import requests
from pprint import pprint

url = "https://tabular-api.data.gouv.fr/api/resources/851d342f-9c96-41c1-924a-11a7a7aae8a6/data/?page=1&page_size=100"

response = requests.get(url)
data = response.json()
pprint(data['data'][:10])

[{'__id': 1,
  'code_geo': '01',
  'code_parent': 'nation',
  'echelle_geo': 'departement',
  'libelle_geo': 'Ain',
  'med_prix_m2_whole_appartement': 2771,
  'med_prix_m2_whole_apt_maison': 2578,
  'med_prix_m2_whole_local': 979,
  'med_prix_m2_whole_maison': 2500,
  'moy_prix_m2_whole_appartement': 3183,
  'moy_prix_m2_whole_apt_maison': 2921,
  'moy_prix_m2_whole_local': 1459,
  'moy_prix_m2_whole_maison': 2757,
  'nb_ventes_whole_appartement': 16522,
  'nb_ventes_whole_apt_maison': 42892,
  'nb_ventes_whole_local': 2133,
  'nb_ventes_whole_maison': 26370},
 {'__id': 2,
  'code_geo': '01001',
  'code_parent': '200069193',
  'echelle_geo': 'commune',
  'libelle_geo': "L'Abergement-Clemenciat",
  'med_prix_m2_whole_appartement': None,
  'med_prix_m2_whole_apt_maison': 2553,
  'med_prix_m2_whole_local': None,
  'med_prix_m2_whole_maison': 2553,
  'moy_prix_m2_whole_appartement': None,
  'moy_prix_m2_whole_apt_maison': 2637,
  'moy_prix_m2_whole_local': None,
  'moy_prix_m2_whole_maison

In [6]:
pprint({"type":"Feature","id":"11001","geometry":{"type":"MultiPolygon","coordinates":[[[[2.5330192,43.2178536],[2.5336283,43.218002],[2.5339391,43.2180626],[2.5343863,43.2181537],[2.5347376,43.2182276],[2.5347739,43.2182363],[2.5348135,43.2182458],[2.5350581,43.218319],[2.5352429,43.2183836],[2.5354184,43.2184507],[2.5355265,43.218501],[2.535599,43.21854],[2.5356622,43.2185899],[2.5357327,43.2186743],[2.5358302,43.2187603],[2.5359996,43.2188832],[2.537478,43.2196249],[2.5385211,43.2199434],[2.5385116,43.2199563],[2.5385514,43.2199684],[2.5412902,43.2208573],[2.5413199,43.2208654],[2.5413514,43.2208754],[2.5418364,43.2210338],[2.542317,43.2211899],[2.5426137,43.2212858],[2.5429283,43.2213916],[2.5435253,43.2215843],[2.5436043,43.2216108],[2.5463902,43.2225128],[2.5465407,43.2225587],[2.5468287,43.2226542],[2.5478522,43.2229888],[2.5492794,43.2234411],[2.5507417,43.2239081],[2.5556377,43.2254911],[2.5557391,43.2255197],[2.5583303,43.2263626],[2.5592479,43.2266604],[2.5619298,43.227519],[2.5620593,43.227562],[2.5621181,43.22758],[2.5622559,43.2276248],[2.5649282,43.228484],[2.5667578,43.2290702],[2.5670107,43.2291433],[2.5672535,43.2292236],[2.5673414,43.2293793],[2.5673118,43.2293953],[2.5672863,43.2294117],[2.5672211,43.2294512],[2.5671111,43.2295175],[2.5670696,43.2295414],[2.5667811,43.2297242],[2.5654984,43.2305314],[2.5641705,43.231358],[2.5633769,43.2318674],[2.5633314,43.2318968],[2.5632126,43.2319744],[2.5631068,43.2320392],[2.5628628,43.2321883],[2.5614568,43.2330683],[2.560073,43.2339342],[2.5588744,43.2346843],[2.5588195,43.2347187],[2.5587557,43.2347587],[2.5556628,43.2366939],[2.5554324,43.2368381],[2.5554,43.2368585],[2.5553663,43.2368794],[2.55465,43.2373281],[2.5545566,43.2373861],[2.5530618,43.2383246],[2.5527751,43.2385052],[2.5520168,43.2389785],[2.5514172,43.239354],[2.551367,43.2393844],[2.5509146,43.2396646],[2.5500094,43.2402308],[2.5495495,43.2405186],[2.5491899,43.2407435],[2.5484493,43.2412067],[2.5484253,43.2412218],[2.5484038,43.2412353],[2.5483086,43.2413023],[2.5483037,43.2412402],[2.5482514,43.2412766],[2.5479925,43.2414334],[2.5478869,43.2414944],[2.5474381,43.2421163],[2.5467766,43.2430343],[2.5467457,43.243074],[2.5465828,43.2432984],[2.5465107,43.2433926],[2.546441,43.2434547],[2.5459734,43.2437177],[2.545953,43.2437273],[2.5459356,43.2437348],[2.5457325,43.2438096],[2.5455903,43.2438604],[2.5453667,43.2439297],[2.5452783,43.2439672],[2.5450989,43.2440256],[2.5450807,43.2440509],[2.5446694,43.2442789],[2.5444592,43.2444448],[2.5440842,43.2445819],[2.5440463,43.2446126],[2.5440227,43.2446684],[2.543986,43.2447426],[2.543864,43.244914],[2.5436386,43.2451541],[2.5433927,43.2452284],[2.5433755,43.2452047],[2.5430523,43.2452756],[2.5428456,43.2453139],[2.5427156,43.2453382],[2.5425116,43.2454311],[2.5422498,43.2455839],[2.5417907,43.2458493],[2.5417026,43.2459332],[2.5416652,43.2459326],[2.5412427,43.2461972],[2.5401782,43.2473496],[2.5397371,43.2474161],[2.5382705,43.2482442],[2.5382117,43.2483115],[2.538146,43.2483682],[2.5380761,43.2484041],[2.5380251,43.2484365],[2.5380222,43.2484644],[2.5378429,43.2484578],[2.5375963,43.2484626],[2.5373113,43.2485241],[2.5361252,43.2489083],[2.5355618,43.2491058],[2.5355893,43.2491233],[2.5358014,43.2493214],[2.5358865,43.2494111],[2.5359965,43.2497538],[2.5360188,43.2498118],[2.5354512,43.2498619],[2.5349835,43.2498705],[2.534699,43.2499013],[2.5345165,43.24998],[2.5344158,43.2500509],[2.5342583,43.2502428],[2.5340945,43.250401],[2.5340485,43.2503914],[2.5334414,43.2501941],[2.5334417,43.2501661],[2.5331381,43.2501972],[2.5328566,43.2502154],[2.5326675,43.2502101],[2.5319768,43.250076],[2.5314187,43.250016],[2.5312295,43.249969],[2.5311469,43.2499207],[2.5308947,43.2498097],[2.5303053,43.2496799],[2.5301486,43.2496537],[2.5299833,43.2496441],[2.5294497,43.2496696],[2.5293615,43.2496756],[2.5283923,43.2497111],[2.5281061,43.2497351],[2.5278141,43.2497149],[2.5268897,43.2496108],[2.5260095,43.2495087],[2.525348,43.2494361],[2.5250662,43.2494121],[2.5245193,43.2493935],[2.5241992,43.2493867],[2.5237537,43.2493906],[2.5228161,43.2494111],[2.5223849,43.2493951],[2.5217804,43.2494125],[2.5217175,43.2494098],[2.521597,43.2493925],[2.5214605,43.2493632],[2.5212473,43.2492942],[2.5211663,43.2492696],[2.521,43.2492626],[2.520879,43.2492612],[2.5207497,43.2492378],[2.5206489,43.2491975],[2.5205746,43.2491544],[2.5204864,43.2491424],[2.5203023,43.2491225],[2.520076,43.2491339],[2.5198318,43.2491578],[2.5195956,43.2491601],[2.5193041,43.2491505],[2.519115,43.2491486],[2.5187965,43.2491668],[2.5183737,43.2492038],[2.5179929,43.2491765],[2.5179946,43.2491564],[2.5175465,43.2491501],[2.5168544,43.2491048],[2.5167872,43.2490957],[2.516136,43.2489705],[2.5160245,43.2489759],[2.5153851,43.2490477],[2.5151841,43.249119],[2.514538,43.2491323],[2.5139683,43.2491293],[2.5136879,43.2491452],[2.5136468,43.2491475],[2.5125821,43.2492049],[2.5123767,43.2491409],[2.5115082,43.2491767],[2.5102599,43.2491833],[2.5095648,43.2491382],[2.5088777,43.249139],[2.5081654,43.2491812],[2.5075199,43.2492313],[2.5075187,43.249203],[2.5075611,43.2488905],[2.507592,43.2483695],[2.5074564,43.2478459],[2.5073181,43.2473145],[2.5072649,43.2467849],[2.5071743,43.2458927],[2.507106,43.2457372],[2.5064205,43.2446148],[2.5063622,43.2444662],[2.5063415,43.2444345],[2.5059488,43.2437142],[2.5055595,43.2430036],[2.5051806,43.2423046],[2.5049504,43.2420585],[2.5046704,43.241708],[2.5046384,43.2416113],[2.5046265,43.2415798],[2.5044401,43.2407952],[2.5044382,43.2407738],[2.5047681,43.2405845],[2.5055761,43.2399424],[2.5058849,43.239701],[2.5069725,43.2386976],[2.508397,43.2372229],[2.5085593,43.2369836],[2.5086847,43.2368103],[2.5088694,43.2365956],[2.509086,43.2364107],[2.5092942,43.2362421],[2.5096331,43.2359448],[2.5097278,43.2358476],[2.5098559,43.2357197],[2.510264,43.2353484],[2.5102602,43.2353238],[2.5098273,43.2343507],[2.5098755,43.2342947],[2.5097949,43.2342042],[2.5094405,43.2334052],[2.5090477,43.2329922],[2.5089622,43.2328918],[2.5088838,43.2327501],[2.5088464,43.2326136],[2.5088499,43.2324847],[2.5089656,43.2315445],[2.509129,43.2304204],[2.509133,43.2304119],[2.5091869,43.2304272],[2.5092983,43.2304493],[2.5093521,43.2304716],[2.5094447,43.2305234],[2.5098336,43.2307011],[2.5099101,43.2307187],[2.5099788,43.2307347],[2.5100692,43.2307661],[2.5101141,43.2307754],[2.510161,43.2307761],[2.5102143,43.2307727],[2.5102541,43.230764],[2.5102791,43.2307687],[2.5105823,43.2308087],[2.5108775,43.2308379],[2.5115069,43.2308816],[2.5115148,43.2309023],[2.5121881,43.2309817],[2.5122883,43.2309453],[2.5124056,43.2309552],[2.5124211,43.2309503],[2.5126858,43.2309361],[2.5131223,43.2309354],[2.5133556,43.2309008],[2.5133815,43.2308923],[2.513447,43.2308033],[2.5134915,43.2307426],[2.513623,43.2307277],[2.5140641,43.2306405],[2.5141563,43.2306393],[2.5142996,43.2306487],[2.5144776,43.2306081],[2.5146926,43.230525],[2.5153485,43.2303595],[2.5156014,43.2303116],[2.5160686,43.2303243],[2.516342,43.2303078],[2.5165236,43.2302863],[2.5168408,43.2302337],[2.5173153,43.2300043],[2.5176259,43.2298804],[2.5180261,43.2298066],[2.5185382,43.2297841],[2.5189153,43.2297638],[2.519219,43.2297375],[2.5194425,43.2296915],[2.5196095,43.2296856],[2.5200144,43.2297315],[2.5203362,43.2297559],[2.5205985,43.2297228],[2.520823,43.2296362],[2.5210043,43.2294963],[2.5211298,43.2294016],[2.5213092,43.2292806],[2.5214972,43.2292105],[2.52166,43.2291506],[2.5213038,43.228628],[2.5212564,43.2286028],[2.5212631,43.2285929],[2.5213301,43.2284501],[2.5212734,43.2284444],[2.5212518,43.228301],[2.5212532,43.2281992],[2.5212666,43.2280957],[2.521291,43.2279944],[2.5213192,43.2279208],[2.5213671,43.2278385],[2.5214725,43.2276971],[2.5214952,43.227654],[2.521486,43.2276161],[2.5214497,43.2275691],[2.5213676,43.2274568],[2.5213213,43.2274151],[2.52123,43.2273121],[2.5212037,43.2272545],[2.5211938,43.2272251],[2.5212033,43.2272007],[2.5212713,43.2271278],[2.5213591,43.2270267],[2.5213841,43.2269985],[2.5214099,43.2269819],[2.5215268,43.2269168],[2.5216628,43.2268266],[2.5217829,43.2267075],[2.5218265,43.2266502],[2.52184,43.2265998],[2.5218552,43.226463],[2.5219289,43.2263628],[2.5222471,43.225306],[2.5222189,43.2251707],[2.5218512,43.2243748],[2.5217928,43.2243018],[2.5217337,43.2242596],[2.5216605,43.2242199],[2.521619,43.2241155],[2.521731,43.2240566],[2.5218862,43.2235608],[2.5218702,43.2234097],[2.5221253,43.223251],[2.5221725,43.2232158],[2.5222484,43.2232458],[2.5222911,43.223215],[2.5223922,43.2231955],[2.5233458,43.223023],[2.5238677,43.2229141],[2.5238681,43.2228862],[2.5240374,43.2228977],[2.5240386,43.2228828],[2.5240452,43.2228664],[2.5250266,43.2219696],[2.5251151,43.2218864],[2.5251338,43.2218607],[2.5257782,43.2211702],[2.5258107,43.221147],[2.525938,43.2209906],[2.5259492,43.2209768],[2.5260363,43.2209996],[2.5261098,43.2209798],[2.5263125,43.2207506],[2.5265925,43.2204203],[2.5267725,43.220103],[2.5269019,43.2198812],[2.5269916,43.2197514],[2.527119,43.2195843],[2.5271344,43.2195592],[2.5271698,43.2194886],[2.5273403,43.2193211],[2.5276314,43.2193459],[2.5285708,43.219423],[2.5287498,43.2194501],[2.5294388,43.2194814],[2.5297503,43.2194881],[2.5299625,43.2194829],[2.5303549,43.2193401],[2.5305643,43.2191899],[2.5307961,43.2190233],[2.5309268,43.2189934],[2.5311225,43.2189804],[2.5311748,43.2189762],[2.5311546,43.2187806],[2.5312405,43.2187027],[2.5313395,43.2186082],[2.5321013,43.2180477],[2.5322007,43.217974],[2.5322794,43.2179472],[2.5327203,43.2178881],[2.5328445,43.2178725],[2.5329702,43.2178624],[2.5330138,43.217866],[2.5330192,43.2178536]]]]},"properties":{"id":"11001","nom":"AIGUES VIVES","created":"2006-09-28","updated":"2025-10-27"}},
)

{'geometry': {'coordinates': [[[[2.5330192, 43.2178536],
                                [2.5336283, 43.218002],
                                [2.5339391, 43.2180626],
                                [2.5343863, 43.2181537],
                                [2.5347376, 43.2182276],
                                [2.5347739, 43.2182363],
                                [2.5348135, 43.2182458],
                                [2.5350581, 43.218319],
                                [2.5352429, 43.2183836],
                                [2.5354184, 43.2184507],
                                [2.5355265, 43.218501],
                                [2.535599, 43.21854],
                                [2.5356622, 43.2185899],
                                [2.5357327, 43.2186743],
                                [2.5358302, 43.2187603],
                                [2.5359996, 43.2188832],
                                [2.537478, 43.2196249],
                                [2.538

In [3]:
import pandas as pd

In [17]:
df = pd.read_csv('https://files.data.gouv.fr/geo-dvf/latest/csv/2025/departements/01.csv.gz', dtype=str)
df[df['nature_mutation'] != 'Vente']

,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude
24,2025-6,2025-01-06,000001,Vente en l'état futur d'achèvement,543510,NaN,NaN,PAIMBOEUF,B005,01210,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.111117,46.252129
25,2025-6,2025-01-06,000001,Vente en l'état futur d'achèvement,543510,NaN,NaN,PAIMBOEUF,B005,01210,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.111117,46.252129
56,2025-17,2025-01-06,000001,Vente en l'état futur d'achèvement,274000,NaN,NaN,LES PLANTEES DE LA MALADIE,B197,01500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.352706,45.964455
57,2025-17,2025-01-06,000001,Vente en l'état futur d'achèvement,274000,NaN,NaN,LES PLANTEES DE LA MALADIE,B197,01500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.352706,45.964455
70,2025-23,2025-01-13,000001,Vente en l'état futur d'achèvement,193500,NaN,NaN,LES PLANTEES DE LA MALADIE,B197,01500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.352706,45.964455
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42289,2025-14070,2025-12-30,000001,Vente en l'état futur d'achèvement,245000,21,NaN,CHE DES MARES,0512,01220,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.134784,46.350981
42291,2025-14072,2025-12-30,000001,Vente en l'état futur d'achèvement,400466,21,NaN,CHE DES MARES,0512,01220,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.134784,46.350981
42292,2025-14072,2025-12-30,000001,Vente en l'état futur d'achèvement,400466,21,NaN,CHE DES MARES,0512,01220,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.134784,46.350981
42300,2025-14076,2025-12-30,000001,Vente en l'état futur d'achèvement,306000,21,NaN,CHE DES MARES,0512,01220,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.134784,46.350981


In [18]:
df.rename(columns={'code_commune': 'insee_code'}, inplace=True) 

In [23]:
df["nombre_pieces_principales"] = df["nombre_pieces_principales"].astype("Int64")

Int64Dtype()